In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from models.Pix2Pix_128_V2 import UnetGeneratorSmall
from utils import calculate_mse, calculate_psnr, calculate_ssim, calculate_correlation

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Przygotowanie danych

In [ ]:
from dival import get_standard_dataset

dataset = get_standard_dataset(
    "custom",
    data_path="/home/kamil/ct_reconstruction_dataset",
    sinogram_shape=(256, 183),
    image_shape=(128, 128),
    parts_len={"train": 206143, "validation": 25767, "test": 25769},
    impl="skimage",
)

train_dataset = dataset.create_torch_dataset(part="train")
val_dataset = dataset.create_torch_dataset(part="validation")
test_dataset = dataset.create_torch_dataset(part="test")

In [ ]:
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

### Model ResUNetA (UnetGeneratorSmall)

In [ ]:
model = UnetGeneratorSmall().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Trainable parameters: {total_params:,}")

### Trening modelu

In [ ]:
def prepare_batch(sino, img):
    """Add channel dim and move to device."""
    sino = sino.unsqueeze(1).to(device, non_blocking=True)
    img = img.unsqueeze(1).to(device, non_blocking=True)
    return sino, img


def save_checkpoint(epoch, model, optimizer):
    os.makedirs("checkpoints", exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        f"checkpoints/resuneta_{epoch}.pt",
    )

In [ ]:
from datetime import date

num_epochs = 50
checkpoint_every = 5
today = date.today().strftime("%m%d")

best_val_mse = float("inf")

train_losses = []
(
    mse_train_losses,
    ssim_train_losses,
    psnr_train_losses,
    correlation_train_losses,
) = ([], [], [], [])
val_losses = []
(
    mse_val_losses,
    ssim_val_losses,
    psnr_val_losses,
    correlation_val_losses,
) = ([], [], [], [])

with open("training_log.txt", "w") as f:
    f.write("")

print("Training started")

for epoch in range(num_epochs):

    # ================= TRAIN =================
    model.train()
    train_loss = 0.0
    mse_train = ssim_train = psnr_train = corr_train = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=False,
    )
    for sino, img in train_bar:
        sino, img = prepare_batch(sino, img)

        output = model(sino)
        loss = criterion(output, img)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        mse_train += calculate_mse(output, img)
        ssim_train += calculate_ssim(output, img)
        psnr_train += calculate_psnr(output, img)
        corr_train += calculate_correlation(output, img)

    n_train = len(train_loader)
    train_loss /= n_train
    mse_train /= n_train
    ssim_train /= n_train
    psnr_train /= n_train
    corr_train /= n_train

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0.0
    mse_val = ssim_val = psnr_val = corr_val = 0.0

    with torch.inference_mode():
        val_bar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{num_epochs} [Val]",
            leave=False,
        )
        for sino, img in val_bar:
            sino, img = prepare_batch(sino, img)

            output = model(sino)
            loss = criterion(output, img)

            val_loss += loss.item()
            mse_val += calculate_mse(output, img)
            ssim_val += calculate_ssim(output, img)
            psnr_val += calculate_psnr(output, img)
            corr_val += calculate_correlation(output, img)

    n_val = len(val_loader)
    val_loss /= n_val
    mse_val /= n_val
    ssim_val /= n_val
    psnr_val /= n_val
    corr_val /= n_val

    # ================= SAVE =================
    if epoch % checkpoint_every == 0:
        save_checkpoint(epoch, model, optimizer)

    if mse_val < best_val_mse:
        best_val_mse = mse_val
        torch.save(model.state_dict(), f"resuneta_{today}_best.pth")

    # ================= STORE HISTORY =================
    train_losses.append(train_loss)
    mse_train_losses.append(mse_train)
    ssim_train_losses.append(ssim_train)
    psnr_train_losses.append(psnr_train)
    correlation_train_losses.append(corr_train)

    val_losses.append(val_loss)
    mse_val_losses.append(mse_val)
    ssim_val_losses.append(ssim_val)
    psnr_val_losses.append(psnr_val)
    correlation_val_losses.append(corr_val)

    # ================= LOG =================
    log = (
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train MSE: {mse_train:.6f} | "
        f"Val MSE: {mse_val:.6f} | "
        f"Train SSIM: {ssim_train:.4f} | "
        f"Val SSIM: {ssim_val:.4f} | "
        f"Train PSNR: {psnr_train:.4f} | "
        f"Val PSNR: {psnr_val:.4f} | "
        f"Train Corr: {corr_train:.4f} | "
        f"Val Corr: {corr_val:.4f}"
    )
    print(log)
    with open(f"training_log{today}.txt", "a") as f:
        f.write(log + "\n")

print("Training completed")

### Wykresy i zapis wag

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(train_losses) + 1)

# Loss
plt.figure()
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses,   label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend(); plt.grid()
plt.show()

# MSE
plt.figure()
plt.plot(epochs_range, mse_train_losses, label="Train MSE")
plt.plot(epochs_range, mse_val_losses,   label="Val MSE")
plt.xlabel("Epoch"); plt.ylabel("MSE")
plt.title("MSE")
plt.legend(); plt.grid()
plt.show()

# SSIM
plt.figure()
plt.plot(epochs_range, ssim_train_losses, label="Train SSIM")
plt.plot(epochs_range, ssim_val_losses,   label="Val SSIM")
plt.xlabel("Epoch"); plt.ylabel("SSIM")
plt.title("SSIM")
plt.legend(); plt.grid()
plt.show()

# PSNR
plt.figure()
plt.plot(epochs_range, psnr_train_losses, label="Train PSNR")
plt.plot(epochs_range, psnr_val_losses,   label="Val PSNR")
plt.xlabel("Epoch"); plt.ylabel("PSNR")
plt.title("PSNR")
plt.legend(); plt.grid()
plt.show()

# Correlation
plt.figure()
plt.plot(epochs_range, correlation_train_losses, label="Train Corr")
plt.plot(epochs_range, correlation_val_losses,   label="Val Corr")
plt.xlabel("Epoch"); plt.ylabel("Correlation")
plt.title("Correlation")
plt.legend(); plt.grid()
plt.show()